# Evaluation Results Visualization

This notebook parses JSON result files from `results/gemini_gemini-3-flash-preview/` (the `log/` subfolder is ignored) and visualizes scores by task and difficulty.

In [1]:
import json
import os
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List
import numpy as np
import matplotlib.font_manager as fm

# Korean font setup (macOS/Linux/Windows)
# Find available Korean fonts
korean_fonts = ['AppleGothic', 'Apple SD Gothic Neo', 'NanumGothic', 'NanumBarunGothic', 'Malgun Gothic', 'Noto Sans CJK KR']
available_font = None
font_list = [f.name for f in fm.fontManager.ttflist]

for font in korean_fonts:
    if font in font_list:
        available_font = font
        break

if available_font:
    plt.rcParams['font.family'] = available_font
    print(f"Korean font set: {available_font}")
else:
    # Fallback: try to find any Korean-supporting font
    korean_support_fonts = [f for f in font_list if any(k in f for k in ['Apple', 'Nanum', 'Gothic', 'Malgun'])]
    if korean_support_fonts:
        # Prefer AppleGothic or NanumGothic
        preferred = [f for f in korean_support_fonts if 'AppleGothic' in f or 'NanumGothic' in f]
        if preferred:
            plt.rcParams['font.family'] = preferred[0]
            print(f"Korean font set (fallback): {preferred[0]}")
        else:
            plt.rcParams['font.family'] = korean_support_fonts[0]
            print(f"Korean font set (fallback): {korean_support_fonts[0]}")
    else:
        plt.rcParams['font.family'] = 'DejaVu Sans'
        print("Warning: No Korean font found. Using default font (Korean may not display correctly)")

# Prevent minus sign from breaking
plt.rcParams['axes.unicode_minus'] = False

sns.set_style("whitegrid")
sns.set_palette("husl")


def _repo_root() -> Path:
    """Resolve repo root so paths work regardless of Jupyter cwd."""
    start = Path.cwd().resolve()
    for p in [start, *start.parents]:
        if (p / "evaluation").is_dir() and (p / "results").is_dir():
            return p
    raise FileNotFoundError(
        "logical-puzzles repo root not found (expected evaluation/ and results/). "
        f"cwd={start}"
    )


# Results: single model run folder (task subdirs only; log/ is excluded in loader)
RESULTS_DIR = _repo_root() / "results" / "gemini_gemini-3-flash-preview"

print("Libraries loaded successfully")

Korean font set: AppleGothic
Libraries loaded successfully


## Data Loading Function

In [2]:
def load_all_results(results_dir: Path) -> pd.DataFrame:
    """Load JSON summaries from results/<model>/ (skips the `log/` directory)."""
    data = []
    model_name = results_dir.name

    for task_dir in results_dir.iterdir():
        if not task_dir.is_dir():
            continue
        if task_dir.name == "log":
            continue

        task_name = task_dir.name

        json_files = list(task_dir.glob("*.json"))
        if not json_files:
            continue

        json_files.sort(key=lambda f: f.name)
        json_files = [json_files[-1]]

        for json_file in json_files:
            try:
                with open(json_file, 'r', encoding='utf-8') as f:
                    result = json.load(f)

                metadata = result.get("metadata", {})
                summary = result.get("summary", {})
                overall = summary.get("overall", {})
                by_difficulty = summary.get("by_difficulty", {})

                data.append({
                    "model": model_name,
                    "task": task_name,
                    "timestamp": metadata.get("timestamp", ""),
                    "total_puzzles": metadata.get("total_puzzles", 0),
                    "accuracy": overall.get("accuracy", 0),
                    "correct_count": overall.get("correct_count", 0),
                    "total_count": overall.get("total_count", 0),
                    "avg_latency_ms": overall.get("avg_latency_ms", 0),
                    "difficulty": "overall"
                })

                for difficulty, stats in by_difficulty.items():
                    data.append({
                        "model": model_name,
                        "task": task_name,
                        "timestamp": metadata.get("timestamp", ""),
                        "total_puzzles": stats.get("total", 0),
                        "accuracy": stats.get("accuracy", 0),
                        "correct_count": stats.get("correct", 0),
                        "total_count": stats.get("total", 0),
                        "avg_latency_ms": overall.get("avg_latency_ms", 0),  # 난이도별 latency는 없으므로 전체 사용
                        "difficulty": difficulty
                    })

            except Exception as e:
                print(f"Error loading {json_file}: {e}")
                continue

    return pd.DataFrame(data)

## Load and Check Data

In [3]:
# Load data
df = load_all_results(RESULTS_DIR)
print(f"Loaded {len(df)} records")
print(f"\nTasks: {sorted(df['task'].unique())}")
print(f"\nModels: {df['model'].unique()}")
print(f"\nDifficulties: {sorted(df['difficulty'].unique())}")

# Filter overall statistics only
df_overall = df[df['difficulty'] == 'overall'].copy()

# Filter difficulty-specific statistics only
df_by_difficulty = df[df['difficulty'] != 'overall'].copy()

print(f"\nOverall statistics records: {len(df_overall)}")
print(f"Difficulty-specific records: {len(df_by_difficulty)}")

FileNotFoundError: [Errno 2] No such file or directory: '/Users/joon/Dev/AI/logical-puzzles/results/gemini_gemini-3-flash-preview'

## 1. Overall Accuracy by Task

In [ ]:
# Compute task-level accuracy as mean over difficulties (easy/medium/hard)
if len(df_by_difficulty) > 0:
    task_mean_df = (
        df_by_difficulty[df_by_difficulty['difficulty'].isin(['easy', 'medium', 'hard'])]
        .groupby('task', as_index=False)['accuracy']
        .mean()
    )
else:
    # Fallback when difficulty rows are unavailable
    task_mean_df = (
        df_overall[['task', 'accuracy']]
        .groupby('task', as_index=False)['accuracy']
        .mean()
    )

plt.figure(figsize=(14, 6))
task_order = task_mean_df.sort_values('accuracy', ascending=False)['task']
ax = sns.barplot(data=task_mean_df, x='task', y='accuracy', order=task_order, color='#4C72B0')
plt.title('Overall Accuracy by Task (Mean over Difficulties)', fontsize=16, fontweight='bold')
plt.xlabel('Task', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.ylim(0, 1.1)

# Add value labels on bars
for container in ax.containers:
    ax.bar_label(container, fmt='%.3f', label_type='edge', padding=3, fontsize=9)

plt.tight_layout()
plt.show()

## 2. Accuracy by Task and Difficulty (Grouped Bar Chart)

In [ ]:
if len(df_by_difficulty) > 0:
    plt.figure(figsize=(16, 6))

    # Keep current style but force 3 grouped bars per task (easy/medium/hard)
    difficulty_order = ['easy', 'medium', 'hard']
    available_difficulties = [d for d in difficulty_order if d in df_by_difficulty['difficulty'].values]

    if available_difficulties:
        # Color palette: easy=green, medium=yellow, hard=pink
        difficulty_colors = {
            'easy': '#2ecc71',      # Green
            'medium': '#f1c40f',    # Yellow
            'hard': '#ff69b4',      # Pink
        }
        palette = [difficulty_colors.get(d, '#3498db') for d in available_difficulties]

        # Normalize task name: e.g., logic_grid_en_easy -> logic_grid_en
        plot_base = df_by_difficulty[
            df_by_difficulty['difficulty'].isin(available_difficulties)
        ].copy()
        plot_base['base_task'] = plot_base['task'].str.replace(
            r'_(easy|medium|hard)$', '', regex=True
        )

        # Aggregate to one value per (base_task, difficulty)
        plot_df = (
            plot_base.groupby(['base_task', 'difficulty'], as_index=False)['accuracy']
            .mean()
        )

        # Task order by mean of available difficulties
        task_order = (
            plot_df.groupby('base_task')['accuracy']
            .mean()
            .sort_values(ascending=False)
            .index
        )

        ax = sns.barplot(
            data=plot_df,
            x='base_task',
            y='accuracy',
            hue='difficulty',
            hue_order=available_difficulties,
            order=task_order,
            palette=palette,
        )
        plt.title('Accuracy by Task and Difficulty', fontsize=16, fontweight='bold')
        plt.xlabel('Task', fontsize=12)
        plt.ylabel('Accuracy', fontsize=12)
        plt.ylim(0, 1.1)
        plt.legend(title='Difficulty', loc='upper right')

        tick_labels = list(task_order)

        from matplotlib.ticker import FixedLocator
        ax.xaxis.set_major_locator(FixedLocator(range(len(task_order))))
        ax.set_xticklabels(tick_labels, rotation=45, ha='right')

        # Add value labels on bars (diagonal on top of bars to avoid overlap)
        for container in ax.containers:
            labels = [f'{val:.2f}' for val in container.datavalues]
            ax.bar_label(container, labels=labels, label_type='edge', padding=3, fontsize=8, rotation=45)

        plt.tight_layout()
        plt.show()
else:
    print("No difficulty-specific data available.")

## 3. Accuracy Heatmap by Task and Difficulty

In [ ]:
if len(df_by_difficulty) > 0:
    pivot_data = df_by_difficulty.pivot_table(
        values='accuracy', 
        index='task', 
        columns='difficulty', 
        aggfunc='mean'
    )
    
    # Reorder columns by difficulty (easy -> medium -> hard -> expert)
    difficulty_order = ['easy', 'medium', 'hard', 'expert']
    existing_difficulties = [d for d in difficulty_order if d in pivot_data.columns]
    pivot_data = pivot_data[existing_difficulties]
    
    task_order = df_overall.groupby('task')['accuracy'].mean().sort_values(ascending=False).index
    pivot_data = pivot_data.reindex(task_order)
    
    plt.figure(figsize=(10, max(6, len(pivot_data) * 0.5)))
    sns.heatmap(pivot_data, annot=True, fmt='.2f', cmap='RdYlGn', 
                vmin=0, vmax=1, cbar_kws={'label': 'Accuracy'}, linewidths=0.5)
    plt.title('Accuracy Heatmap by Task and Difficulty', fontsize=16, fontweight='bold')
    plt.xlabel('Difficulty', fontsize=12)
    plt.ylabel('Task', fontsize=12)
    plt.tight_layout()
    plt.show()
else:
    print("No difficulty-specific data available.")

## 4. Average Latency by Task

In [ ]:
# Average latency by task
plt.figure(figsize=(14, 6))
task_order = df_overall.groupby('task')['avg_latency_ms'].mean().sort_values(ascending=False).index
ax = sns.barplot(data=df_overall, x='task', y='avg_latency_ms', order=task_order)
plt.title('Average Latency by Task', fontsize=16, fontweight='bold')
plt.xlabel('Task', fontsize=12)
plt.ylabel('Average Latency (ms)', fontsize=12)
plt.xticks(rotation=45, ha='right')

# Add value labels on bars (format as thousands with 'k' suffix for readability)
for container in ax.containers:
    labels = []
    for val in container.datavalues:
        if val >= 1000:
            labels.append(f'{val/1000:.1f}k')
        else:
            labels.append(f'{val:.0f}')
    # Use 'edge' with top padding to place labels above bars
    ax.bar_label(container, labels=labels, label_type='edge', padding=5, fontsize=9, rotation=0)

plt.tight_layout()
plt.show()

## 5. Accuracy vs Latency Scatter Plot

In [ ]:
plt.figure(figsize=(10, 6))
scatter = plt.scatter(df_overall['avg_latency_ms'], df_overall['accuracy'], 
                     s=100, alpha=0.6, c=range(len(df_overall)), cmap='viridis')
for i, row in df_overall.iterrows():
    plt.annotate(row['task'], (row['avg_latency_ms'], row['accuracy']), 
                fontsize=8, alpha=0.7)
plt.xlabel('Average Latency (ms)', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('Accuracy vs Latency by Task', fontsize=16, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## View detailed data

In [ ]:
# Overall data preview
print("Overall Data Preview:")
print(df_overall[['task', 'model', 'accuracy', 'avg_latency_ms', 'total_count']].to_string())

# Task-specific detailed information
if len(df_overall) > 0:
    print("\n\nTask-specific Details:")
    for task in sorted(df_overall['task'].unique()):
        task_data = df_overall[df_overall['task'] == task]
        print(f"\n{task}:")
        print(f"  Accuracy: {task_data['accuracy'].mean():.3f} (std: {task_data['accuracy'].std():.3f})")
        print(f"  Latency: {task_data['avg_latency_ms'].mean():.1f}ms (std: {task_data['avg_latency_ms'].std():.1f}ms)")
        print(f"  Total puzzles: {task_data['total_count'].sum()}")
        
# Difficulty-specific summary
if len(df_by_difficulty) > 0:
    print("\n\nDifficulty-specific Summary:")
    print(df_by_difficulty.groupby(['task', 'difficulty'])[['accuracy', 'correct_count', 'total_count']].first().to_string())